# 13 — Custom Hashing

## Learning goals
- Understand what a hash function does.
- Design simple hash-based signatures for problem-specific data.
- Use tuples and immutable representations as dictionary keys.
- Understand collisions and why equality checks matter.
- Learn rolling hash as a bridge to string matching.

**Important distinction:** In normal Python application code, use built-in `dict` and `set` unless you have a strong reason to implement hashing yourself. In interview problems, “custom hashing” often means designing a compact signature or key.

## 1. What is hashing?

A hash function maps a value to an integer-like hash code.

A hash table uses that hash code to decide where to store or find a key. Python dictionaries and sets use hashing internally.

Conceptually:
1. Compute a hash for the key.
2. Use it to find a candidate storage location.
3. Check equality to confirm the actual key.

A hash value is not the original value, and different values can have the same hash.

In [1]:
print(hash("python"))
print(hash((1, 2, 3)))

# Equal immutable values have equal hashes:
print(hash((1, 2)) == hash((1, 2)))  # True

8951166710975571547
529344067295497451
True


## 2. Hashability in Python

A dictionary key or set element must be hashable:
- Its hash must remain stable while it is used as a key.
- Equality behavior must be consistent with hashing.

Common hashable values: `int`, `str`, `tuple` (if all its members are hashable), `frozenset`.

Common unhashable values: `list`, `dict`, `set`.

A tuple containing a list is also unhashable.

In [2]:
valid = {
    "name": "Ada",
    (1, 2): "coordinate",
    frozenset({2, 3}): "unordered pair"
}
print(valid)

# These raise TypeError:
# hash([1, 2])
# hash({"a": 1})
# hash((1, [2, 3]))

{'name': 'Ada', (1, 2): 'coordinate', frozenset({2, 3}): 'unordered pair'}


## 3. Custom signatures — the key idea

Many interview problems need a key that represents some property of an object.

Examples:
- Anagrams → sorted characters or character-frequency tuple.
- Grid shape → normalized relative coordinates.
- Tree structure → serialized traversal.
- Repeated patterns → prefix or rolling hash.

A good signature has this property:

**Objects that should be considered equivalent produce the same signature.**

It should also avoid treating non-equivalent objects as equal.

## 4. Anagram signature using sorted characters

Two words are anagrams when their sorted characters are identical.

The sorted tuple is immutable, so it can be a dictionary key.

In [3]:
from collections import defaultdict

def group_anagrams(words: list[str]) -> list[list[str]]:
    groups = defaultdict(list)

    for word in words:
        signature = tuple(sorted(word))
        groups[signature].append(word)

    return list(groups.values())

print(group_anagrams(["eat", "tea", "tan", "ate", "nat", "bat"]))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


## 5. Anagram signature using frequency counts

If the input is guaranteed to contain only lowercase English letters, a 26-count tuple is another signature.

Why a tuple? A list is mutable and unhashable; a tuple is immutable and hashable.

This version takes `O(L + 26)` time per word of length `L`.

In [4]:
from collections import defaultdict

def group_anagrams_by_count(words: list[str]) -> list[list[str]]:
    groups = defaultdict(list)

    for word in words:
        count = [0] * 26

        for ch in word:
            count[ord(ch) - ord('a')] += 1

        groups[tuple(count)].append(word)

    return list(groups.values())

print(group_anagrams_by_count(["eat", "tea", "tan", "ate", "nat", "bat"]))

[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


## 6. Design a signature for a 2D shape

Suppose a shape is represented by a list of grid coordinates. Two shapes should match if one is only translated relative to the other.

Normalize every coordinate by subtracting the minimum row and minimum column. The normalized coordinate set becomes a translation-independent signature.

This is a common idea in grid-island shape problems.

In [5]:
def shape_signature(cells: list[tuple[int, int]]) -> tuple:
    min_row = min(r for r, c in cells)
    min_col = min(c for r, c in cells)

    normalized = sorted(
        (r - min_row, c - min_col)
        for r, c in cells
    )

    return tuple(normalized)

shape_a = [(2, 3), (2, 4), (3, 3)]
shape_b = [(7, 8), (7, 9), (8, 8)]

print(shape_signature(shape_a))
print(shape_signature(shape_b))
print(shape_signature(shape_a) == shape_signature(shape_b))  # True

((0, 0), (0, 1), (1, 0))
((0, 0), (0, 1), (1, 0))
True


## 7. Tree serialization as a signature

A tree can be represented by a traversal that includes explicit markers for missing children.

Without null markers, different tree structures can produce the same value sequence. Include a marker such as `#` for every missing child.

In [6]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def serialize(root: TreeNode | None) -> tuple:
    result = []

    def dfs(node):
        if node is None:
            result.append("#")
            return

        result.append(str(node.val))
        dfs(node.left)
        dfs(node.right)

    dfs(root)
    return tuple(result)

tree = TreeNode(1, TreeNode(2), TreeNode(3))
print(serialize(tree))

('1', '2', '#', '#', '3', '#', '#')


## 8. A simple polynomial rolling hash

For a string `s`, a polynomial hash can be defined as:

\[
H(s) = (s_0 B^{n-1} + s_1 B^{n-2} + \dots + s_{n-1}) \bmod M
\]

Where:
- `B` is a chosen base.
- `M` is a modulus.
- Each character is mapped to an integer.

The modulo keeps values bounded. Different strings can still collide, so a hash match is a candidate match, not proof of equality.

In [7]:
def polynomial_hash(s: str, base: int = 31, mod: int = 1_000_000_007) -> int:
    value = 0

    for ch in s:
        value = (value * base + ord(ch)) % mod

    return value

print(polynomial_hash("hello"))
print(polynomial_hash("world"))

99162322
113318802


## 9. Rolling hash — update a window efficiently

For a fixed-length window of length `m`, using the polynomial form:

`H = c0 * B^(m-1) + c1 * B^(m-2) + ... + c(m-1)`

When the window moves:
1. Remove the outgoing character's highest-power contribution.
2. Multiply the remaining hash by `B`.
3. Add the incoming character.
4. Apply modulo `M`.

Formula:

\[
H_{next} = ((H - outgoing \cdot B^{m-1}) \cdot B + incoming) \bmod M
\]

This allows constant-time hash updates per shift after the initial window is built.

In [8]:
def rolling_hash_values(s: str, window_size: int, base: int = 31,
                        mod: int = 1_000_000_007) -> list[int]:
    n = len(s)
    m = window_size

    if not 1 <= m <= n:
        return []

    highest_power = pow(base, m - 1, mod)
    current = 0

    for ch in s[:m]:
        current = (current * base + ord(ch)) % mod

    hashes = [current]

    for right in range(m, n):
        outgoing = ord(s[right - m])
        incoming = ord(s[right])

        current = (current - outgoing * highest_power) % mod
        current = (current * base + incoming) % mod
        hashes.append(current)

    return hashes

print(rolling_hash_values("abcdef", 3))

[96354, 97347, 98340, 99333]


## 10. Rabin–Karp substring search

Rabin–Karp uses rolling hashes to search for a pattern:
1. Hash the pattern.
2. Hash the first text window of the same length.
3. Compare hashes for each window.
4. When hashes match, verify the actual substring to protect against collisions.

Expected performance is often close to `O(n + m)`, but collision behavior and verification can affect worst-case performance.

In [9]:
def rabin_karp(text: str, pattern: str) -> list[int]:
    n, m = len(text), len(pattern)

    if m == 0:
        return list(range(n + 1))
    if m > n:
        return []

    base = 256
    mod = 1_000_000_007
    highest_power = pow(base, m - 1, mod)

    pattern_hash = 0
    window_hash = 0

    for i in range(m):
        pattern_hash = (pattern_hash * base + ord(pattern[i])) % mod
        window_hash = (window_hash * base + ord(text[i])) % mod

    answer = []

    for start in range(n - m + 1):
        if window_hash == pattern_hash:
            if text[start:start + m] == pattern:
                answer.append(start)

        if start < n - m:
            outgoing = ord(text[start])
            incoming = ord(text[start + m])

            window_hash = (window_hash - outgoing * highest_power) % mod
            window_hash = (window_hash * base + incoming) % mod

    return answer

print(rabin_karp("ababcabcabababd", "ababd"))  # [10]
print(rabin_karp("aaaaa", "aa"))               # [0, 1, 2, 3]

[10]
[0, 1, 2, 3]


## 11. Collision handling — why equality still matters

A collision happens when two different keys produce the same hash value.

Hash tables handle this internally by resolving collisions and checking key equality. Therefore:
- Never assume equal hash values mean equal objects.
- Do not write application logic that treats a hash as a unique identifier.
- For rolling-hash algorithms, verify the substring after a hash match when correctness must be guaranteed.

In [10]:
# Python allows different values to share a hash.
# A dictionary still distinguishes them using equality.
class ConstantHash:
    def __init__(self, value):
        self.value = value

    def __hash__(self):
        return 1

    def __eq__(self, other):
        return isinstance(other, ConstantHash) and self.value == other.value

table = {ConstantHash("alpha"): 10, ConstantHash("beta"): 20}
print(len(table))  # 2
print(table[ConstantHash("alpha")])  # 10

2
10


## 12. How to design a custom signature in an interview

Use this process:
1. Define exactly when two inputs should be considered equivalent.
2. Identify which properties must be preserved.
3. Remove irrelevant differences (such as translation, if allowed).
4. Build a deterministic representation.
5. Make it immutable if it will be a dictionary key.
6. Check whether two non-equivalent inputs can share a signature.
7. If using a numeric hash, account for collisions.

A signature can be a tuple or string; it does not need to be a numeric hash.

## 13. Complexity summary

| Technique | Typical cost |
|---|---|
| Sort-based anagram signature | `O(L log L)` per word |
| 26-count signature | `O(L + 26)` per word |
| Normalize grid shape | `O(C log C)` for `C` cells due to sorting |
| Tree serialization | `O(N)` time and `O(N)` output |
| Initial rolling hash | `O(M)` |
| Each rolling update | `O(1)` |
| Rabin–Karp | Expected `O(N + M)`; verification can increase worst-case work |

The signature's storage size also matters. A 26-element tuple uses constant-size structure with respect to word length, while a sorted-character signature grows with the word.

## 14. Common mistakes

1. Trying to use a list as a dictionary key.
2. Assuming a hash is unique.
3. Forgetting equality verification after a rolling-hash match.
4. Designing a signature that loses important information.
5. Normalizing shapes without sorting their coordinates.
6. Forgetting null markers when serializing trees.
7. Using a fixed alphabet signature when the input alphabet is unrestricted.
8. Confusing a problem-specific signature with Python's internal hash implementation.

## 15. Practice questions

1. [49. Group Anagrams](https://leetcode.com/problems/group-anagrams/) — canonical signature.
2. [694. Number of Distinct Islands](https://leetcode.com/problems/number-of-distinct-islands/) — normalized shape signature (Premium).
3. [652. Find Duplicate Subtrees](https://leetcode.com/problems/find-duplicate-subtrees/) — tree serialization/signature.
4. [28. Find the Index of the First Occurrence in a String](https://leetcode.com/problems/find-the-index-of-the-first-occurrence-in-a-string/) — string matching; compare built-in/KMP/Rabin–Karp ideas.
5. [1044. Longest Duplicate Substring](https://leetcode.com/problems/longest-duplicate-substring/) — advanced rolling hash + binary search.

For every problem, explain what the signature preserves and why equal signatures imply the required equivalence.

## 16. Reusable templates

### Immutable frequency signature
```python
def lowercase_signature(s: str) -> tuple[int, ...]:
    count = [0] * 26
    for ch in s:
        count[ord(ch) - ord('a')] += 1
    return tuple(count)
```

### Rolling hash update
```python
# current = hash of current fixed-size window
current = (current - outgoing * highest_power) % mod
current = (current * base + incoming) % mod
```

### Design checklist
```text
What makes two inputs equivalent?
Which details must the signature preserve?
Can the signature be immutable?
Can collisions happen?
Do I need to verify equality after a hash match?
```

This completes the custom-signature and rolling-hash foundation.